
RAG PIPELINE - Data ingestion to Vector db pipeline

In [ ]:
from __future__ import annotations

from typing import Any
import uuid

import chromadb
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import os
from pathlib import Path
import numpy as np

from sentence_transformers import SentenceTransformer

In [ ]:
# reading all PDFs under data/pdf (works whether cwd is repo root or notebook/)
def pdf_data_dir() -> Path:
    for candidate in (Path("data/pdf"), Path("../data/pdf")):
        if candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "No PDF folder found. Expected data/pdf at repo root or ../data/pdf from notebook/."
    )


def load_pdf_files(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)
    print(f"Loading PDF files from {pdf_dir}")
    for file in sorted(pdf_dir.glob("*.pdf")):
        print(f"Loading {file}")
        loader = PyPDFLoader(str(file))
        documents = loader.load()
        for doc in documents:
            doc.metadata["source"] = str(file)
            doc.metadata["type"] = "pdf"
        all_documents.extend(documents)
        print(f"Loaded {len(documents)} documents from {file}")
    return all_documents


all_documents = load_pdf_files(pdf_data_dir())


In [ ]:
all_documents

In [ ]:
# Text splitting into chunks (page-level docs from loader; no merging)
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return text_splitter.split_documents(documents)


if "all_documents" not in globals():
    raise NameError("No documents loaded yet. Run the PDF loading cell first to create all_documents.")

split_docs = split_documents(all_documents)
print(f"Split {len(split_docs)} chunks from {len(all_documents)} page-level PDF document(s)")

if split_docs:
    print("Sample chunk:")
    print("content:", split_docs[0].page_content)
    print("Metadata:", split_docs[0].metadata)


Embedding and vectordb

In [ ]:
#modular programming
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    def __init__(self,model_name:str='all-MiniLM-L6-v2'):
        self.model_name=model_name
        self.model=None
        self._load_model()
    def _load_model(self):
        try:
            self.model=SentenceTransformer(self.model_name)
            print(f"Model {self.model_name} loaded successfully with the dimension {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: list[str]) -> np.ndarray:
        if self.model is None:
            raise ValueError("Model not loaded. Please call _load_model() first.")
        print(f"Generating embeddings for {len(texts)} texts")
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        print(f"Embeddings generated successfully with shape {embeddings.shape}")
        return embeddings

#initilize the class
embedder=EmbeddingManager()
embedder




VectorStore

In [ ]:
# vectorstore
def vector_store_dir() -> str:
    """Resolve vector store path whether cwd is repo root or notebook/."""
    if Path("data/pdf").is_dir():
        path = Path("data/vector_store")
    else:
        path = Path("../data/vector_store")
    path.mkdir(parents=True, exist_ok=True)
    return str(path.resolve())


class VectorStoreManager:
    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str | None = None,
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory or vector_store_dir()
        self.client = None
        self.collection = None
        self._initialize_vectorstore()

    def _initialize_vectorstore(self):
        try:
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "Vector store for PDF documents",
                    "hnsw:space": "cosine",
                },
            )
            print("collection initialized successfully, collection name:", self.collection_name)
            print("Existing documents in collection:", self.collection.count())
        except Exception as e:
            print(f"Error initializing vectorstore: {e}")
            raise

    def add_documents(self, documents: list[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Documents and embeddings must have the same length")

        print(f"Adding {len(documents)} documents to vectorstore")

        ids: list[str] = []
        metadatas: list[dict[str, Any]] = []
        documents_list: list[str] = []
        embeddings_list: list[list[float]] = []

        for i, doc in enumerate(documents):
            ids.append(str(uuid.uuid4()))

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_list.append(doc.page_content)
            embeddings_list.append(embeddings[i].tolist())

        self.collection.add(
            ids=ids,
            documents=documents_list,
            embeddings=embeddings_list,
            metadatas=metadatas,
        )

        print(f"Successfully added documents {len(documents)} to vectorstore")
        print("total documents in vectorstore:", self.collection.count())


# initialize vectorstore
vectorstore = VectorStoreManager()
vectorstore                   
        

In [ ]:
split_docs

In [ ]:
### Convert the text to embeddings
texts=[doc.page_content for doc in split_docs]

## Generate the Embeddings

embeddings=embedder.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(split_docs,embeddings)

### Retriever Pipeline From VectorStore

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStoreManager, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self, query: str, top_k: int = 5, score_threshold: float = 0.0
    ) -> list[dict[str, Any]]:
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        retrieved_docs: list[dict[str, Any]] = []

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # Chroma default is L2; SentenceTransformer vectors are unit-normalized
                    similarity_score = 1.0 - (distance**2) / 2.0
                    if similarity_score >= score_threshold:
                        retrieved_docs.append(
                            {
                                "id": doc_id,
                                "content": document,
                                "metadata": metadata,
                                "similarity_score": similarity_score,
                                "distance": distance,
                                "rank": i + 1,
                            }
                        )

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


rag_retriever = RAGRetriever(vectorstore, embedder)

In [ ]:
rag_retriever

In [ ]:
rag_retriever.retrieve("What is attention is all you need")

In [ ]:
rag_retriever.retrieve("What is claude used for")

In [ ]:
rag_retriever.retrieve("What is claude uescases?")

In [ ]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")

### RAG Pipeline- VectorDB To LLM Output Generation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Do not print secrets. Only check if the key is set.
print("GROQ_API_KEY set:", bool(os.getenv("GROQ_API_KEY")))

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, AIMessage

In [ ]:
class GroqLLM:
    def __init__(self, model_name: str = "llama-3.1-8b-instant", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    


### Advanced RAG (sources, confidence, citations, streaming, history)

In [ ]:
from pathlib import Path
from typing import Any

from langchain_core.messages import HumanMessage


def _doc_source(meta: dict) -> str:
    raw = meta.get("source_file") or meta.get("source", "unknown")
    return Path(raw).name if raw != "unknown" else raw


def _doc_preview(text: str, max_len: int = 300) -> str:
    return text if len(text) <= max_len else text[:max_len] + "..."


def _build_sources(results: list[dict[str, Any]], preview_len: int = 300) -> list[dict[str, Any]]:
    return [
        {
            "source": _doc_source(doc["metadata"]),
            "page": doc["metadata"].get("page", doc["metadata"].get("page_label", "unknown")),
            "score": doc["similarity_score"],
            "preview": _doc_preview(doc["content"], preview_len),
        }
        for doc in results
    ]


def _invoke_llm(groq_llm: GroqLLM, prompt: str) -> str:
    response = groq_llm.llm.invoke([HumanMessage(content=prompt)])
    return response.content


def _stream_llm(groq_llm: GroqLLM, prompt: str) -> str:
    print("Streaming answer:")
    parts: list[str] = []
    for chunk in groq_llm.llm.stream([HumanMessage(content=prompt)]):
        text = chunk.content or ""
        if text:
            print(text, end="", flush=True)
            parts.append(text)
    print()
    return "".join(parts)


def rag_advanced(
    query: str,
    retriever: RAGRetriever,
    llm: GroqLLM,
    top_k: int = 5,
    min_score: float = 0.2,
    return_context: bool = False,
) -> dict[str, Any]:
    """RAG with answer, sources, confidence, and optional full context."""
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        out = {"answer": "No relevant context found.", "sources": [], "confidence": 0.0}
        if return_context:
            out["context"] = ""
        return out

    context = "\n\n".join(doc["content"] for doc in results)
    sources = _build_sources(results)
    confidence = max(doc["similarity_score"] for doc in results)

    prompt = (
        "Use the following context to answer the question concisely.\n"
        f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    )
    answer = _invoke_llm(llm, prompt)

    output: dict[str, Any] = {"answer": answer, "sources": sources, "confidence": confidence}
    if return_context:
        output["context"] = context
    return output


class AdvancedRAGPipeline:
    """RAG with citations, optional streaming, summarization, and query history."""

    def __init__(self, retriever: RAGRetriever, llm: GroqLLM):
        self.retriever = retriever
        self.llm = llm
        self.history: list[dict[str, Any]] = []

    def query(
        self,
        question: str,
        top_k: int = 5,
        min_score: float = 0.2,
        stream: bool = False,
        summarize: bool = False,
    ) -> dict[str, Any]:
        results = self.retriever.retrieve(
            question, top_k=top_k, score_threshold=min_score
        )
        if not results:
            answer = "No relevant context found."
            sources: list[dict[str, Any]] = []
        else:
            context = "\n\n".join(doc["content"] for doc in results)
            sources = _build_sources(results, preview_len=120)
            prompt = (
                "Use the following context to answer the question concisely.\n"
                f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
            )
            answer = _stream_llm(self.llm, prompt) if stream else _invoke_llm(self.llm, prompt)

        citations = [
            f"[{i + 1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)
        ]
        answer_with_citations = (
            answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer
        )

        summary = None
        if summarize and answer and answer != "No relevant context found.":
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary = _invoke_llm(self.llm, summary_prompt)

        self.history.append(
            {
                "question": question,
                "answer": answer,
                "sources": sources,
                "summary": summary,
            }
        )

        return {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary,
            "history": self.history,
        }

In [ ]:
#object
llm=GroqLLM()
llm.generate_response("What is attention is all you need", "This is a test context")

In [ ]:
# Example: rag_advanced
result = rag_advanced(
    "Hard Negative Mining Techniques",
    rag_retriever,
    llm,
    top_k=3,
    min_score=0.1,
    return_context=True,
)
print("Answer:", result["answer"])
print("Sources:", result["sources"])
print("Confidence:", result["confidence"])
print("Context preview:", result["context"][:300])

In [ ]:
# Example: AdvancedRAGPipeline (streaming + summary + citations)
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query(
    "what is attention is all you need",
    top_k=3,
    min_score=0.1,
    stream=True,
    summarize=True,
)
print("\nFinal answer:", result["answer"])
print("Summary:", result["summary"])
print("Last history entry:", result["history"][-1])